# Denoising and Variational Autoencoders with Keras

The previous notebook trained autoencoders to rebuild their input. This one first changes the task: recovering a clean image from a noisy version of it. Then it changes the model, to be able to generate new images. The data is Fashion-MNIST's: 60,000 training and 10,000 test images, 28 × 28 in grayscale like MNIST, but of 10 kinds of clothes and accessories.

1. Corrupting the images
2. A dense denoiser
3. An overcomplete code
4. A convolutional denoiser
5. A variational autoencoder
6. Generating images
7. The effect of β

Each part reuses the data and models of the previous ones. No GPU needed: the notebook runs in twenty minutes to half an hour on a CPU, a quarter of it for part 4's convolutional denoiser. A GPU (`Runtime > Change runtime type`) speeds it up a lot.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers, ops

keras.utils.set_random_seed(0)

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = [
  "T-shirt",
  "Trouser",
  "Pullover",
  "Dress",
  "Coat",
  "Sandal",
  "Shirt",
  "Sneaker",
  "Bag",
  "Ankle boot",
]
input_dim = 28 * 28
# Pixels in [0, 1], like a sigmoid's output
X_train = X_train.reshape(-1, input_dim) / 255
X_test = X_test.reshape(-1, input_dim) / 255
print(X_train.shape, X_test.shape)


def show_images(*rows: np.ndarray, titles: list | np.ndarray | None = None) -> None:
  # Shows rows of 28 × 28 images, one row per array
  n = len(rows[0])
  _, axes = plt.subplots(
    len(rows), n, figsize=(n * 1.2, len(rows) * 1.3), squeeze=False
  )
  for row, images in zip(axes, rows):
    for ax, image in zip(row, images):
      ax.imshow(np.reshape(image, (28, 28)), cmap="gray_r")
      ax.axis("off")
  for ax, title in zip(axes[0], [] if titles is None else titles):
    ax.set_title(title, fontsize=8)
  plt.show()


show_images(X_train[:10], titles=[class_names[y] for y in y_train[:10]])

## 1. Corrupting the images

A denoiser learns to recover the clean image from a damaged version of it: its training examples are pairs (noisy image, clean image). Make the noisy images:

- `X_train_noisy` and `X_test_noisy`: `X_train` and `X_test`, plus a Gaussian noise of mean 0 and standard deviation 0.5 drawn independently for each pixel;
- then bring each pixel back into [0, 1], like those of a real image.

The cell after the solution shows noisy images above the clean ones.

**Useful documentation**

- [`numpy.random.Generator.normal`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html)
- [`numpy.clip`](https://numpy.org/doc/stable/reference/generated/numpy.clip.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `rng = np.random.default_rng(0)`, then `rng.normal(0, 0.5, size=X_train.shape)`: a noise of the images' shape, one draw per pixel
- `np.clip(X, 0, 1)`: the values below 0 become 0, those above 1 become 1
- check: `X_train_noisy.min()` is 0, `X_train_noisy.max()` is 1, and `X_train_noisy.shape` is `X_train`'s

A skeleton to copy into the exercise cell, replacing each `...`:

```python
rng = np.random.default_rng(0)
X_train_noisy = np.clip(X_train + rng.normal(...), ..., ...)
X_test_noisy = ...
```

### Solution

In [ ]:
rng = np.random.default_rng(0)
X_train_noisy = np.clip(X_train + rng.normal(0, 0.5, size=X_train.shape), 0, 1)
X_test_noisy = np.clip(X_test + rng.normal(0, 0.5, size=X_test.shape), 0, 1)

### The noisy images

In [ ]:
titles = [class_names[y] for y in y_test[:10]]
show_images(X_test_noisy[:10], X_test[:10], titles=titles)

## 2. A dense denoiser

Build `autoencoder`, a dense denoiser with one hidden layer:

- a `Dense` layer of `encoding_dim = 12` units with the ReLU activation, the code, on inputs of size `input_dim`;
- the output layer, of `input_dim` units, with an activation suited to pixels in [0, 1];
- compiled with the Adam optimizer and the `"mse"` loss.

The cell after the solution trains it on the pairs (noisy image, clean image), then shows noisy test images, their denoised version and the clean image.

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- a single `keras.Sequential` is enough: `keras.Input(shape=(input_dim,))`, then the two `Dense` layers
- the output: `activation="sigmoid"`, which returns values in ]0, 1[, like the pixels
- `autoencoder.summary()`: 9,420 parameters for the code's layer (784 × 12 + 12) and 10,192 for the output

A skeleton to copy into the exercise cell, replacing each `...`:

```python
encoding_dim = 12
autoencoder = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(...), layers.Dense(...)],
  name="autoencoder",
)
autoencoder.compile(...)
autoencoder.summary()
```

### Solution

In [ ]:
encoding_dim = 12
autoencoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(encoding_dim, activation="relu"),
    layers.Dense(input_dim, activation="sigmoid"),
  ],
  name="autoencoder",
)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

### Training

The input is the noisy image, the target the clean one. Top: the noisy images, then their denoised version, then the clean images.

In [ ]:
autoencoder.fit(
  X_train_noisy,
  X_train,
  epochs=20,
  batch_size=256,
  validation_data=(X_test_noisy, X_test),
  verbose=2,
)
X_test_denoised = autoencoder.predict(X_test_noisy, verbose=0)
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

## 3. An overcomplete code

A code of 12 numbers limits what the denoiser can give back. With a code as large as the input, or larger, an ordinary autoencoder has nothing left to summarize: it can just copy its input (the "Undercomplete or overcomplete code" slide). Does a denoiser do so too? Compare them:

- write `build_dense_autoencoder(encoding_dim)`, which returns part 2's model, compiled the same way, for a given code size;
- train two of them with a code of 1,024 units, more than the 784 pixels, for 10 epochs each: `plain`, an ordinary autoencoder, on the pairs (`X_train`, `X_train`), and `denoiser`, a denoiser, on the pairs (`X_train_noisy`, `X_train`);
- apply both to the noisy test images, `X_test_noisy`, give each one's mean squared error against the clean images, and show the noisy images, `plain`'s output, `denoiser`'s, then the clean images.

Which one copies the noise? Why doesn't the other one, although it would have room to?

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- the body of `build_dense_autoencoder` is part 2's solution, with `encoding_dim` as a parameter, followed by `return autoencoder`; `build_dense_autoencoder(1024).summary()`: 1,607,440 parameters (784 × 1,024 + 1,024, then 1,024 × 784 + 784)
- `plain.fit(X_train, X_train, epochs=10, batch_size=256, verbose=0)`: the target is the input; for `denoiser`, the input is `X_train_noisy`, the target `X_train`. Each one takes a few minutes
- `X_test_plain = plain.predict(X_test_noisy, verbose=0)`, then `np.mean((X_test_plain - X_test) ** 2)`: the error against the clean images

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def build_dense_autoencoder(encoding_dim: int) -> keras.Model:
  autoencoder = ...
  autoencoder.compile(...)
  return autoencoder


plain = build_dense_autoencoder(1024)
plain.fit(...)
denoiser = ...
denoiser.fit(...)
X_test_plain = plain.predict(X_test_noisy, verbose=0)
X_test_denoised = ...
print(f"Errors: ordinary {...:.4f}, denoiser {...:.4f}")
show_images(X_test_noisy[:10], ..., ..., X_test[:10], titles=titles)
```

### Solution

In [ ]:
def build_dense_autoencoder(encoding_dim: int) -> keras.Model:
  autoencoder = keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(encoding_dim, activation="relu"),
      layers.Dense(input_dim, activation="sigmoid"),
    ],
    name="autoencoder",
  )
  autoencoder.compile(optimizer="adam", loss="mse")
  return autoencoder


plain = build_dense_autoencoder(1024)  # more than the 784 pixels
plain.fit(X_train, X_train, epochs=10, batch_size=256, verbose=0)
denoiser = build_dense_autoencoder(1024)
denoiser.fit(X_train_noisy, X_train, epochs=10, batch_size=256, verbose=0)
X_test_plain = plain.predict(X_test_noisy, verbose=0)
X_test_denoised = denoiser.predict(X_test_noisy, verbose=0)
print(f"Ordinary autoencoder: error {np.mean((X_test_plain - X_test) ** 2):.4f}")
print(f"Denoiser: error {np.mean((X_test_denoised - X_test) ** 2):.4f}")
# At the top the noisy images, then the ordinary AE, the denoiser, the clean images
show_images(
  X_test_noisy[:10],
  X_test_plain[:10],
  X_test_denoised[:10],
  X_test[:10],
  titles=titles,
)

The ordinary autoencoder copies the noise: it only learned to return its input, and its code, larger than the image, lets it return it almost as is, noise included. Its error against the clean images is thus much higher. The denoiser, with the same code, returns clean images, sharper than part 2's: its target is the clean image, and each pixel of noise copied would increase its loss. That is what lets a denoiser have an overcomplete code without learning the identity, unlike an ordinary autoencoder.

## 4. A convolutional denoiser

A dense layer handles each pixel on its own; a convolution uses each pixel's neighborhood, which suits an image better. Convolution layers expect a 28 × 28 × 1 image, with an axis for the channels (a single one in grayscale, three in color): the next cell puts the images back into that shape.

In [ ]:
X_train_img = X_train.reshape(-1, 28, 28, 1)
X_test_img = X_test.reshape(-1, 28, 28, 1)
X_train_noisy_img = X_train_noisy.reshape(-1, 28, 28, 1)
X_test_noisy_img = X_test_noisy.reshape(-1, 28, 28, 1)
print(X_train_img.shape)

### The model

Build a fully convolutional `encoder` and `decoder`, then `autoencoder`, which chains them. All the `Conv2D` have 3 × 3 filters and `padding="same"`:

- `encoder`, on 28 × 28 × 1 inputs: a `Conv2D` of 16 filters (ReLU) and a 2 × 2 `MaxPooling2D`, then a `Conv2D` of 8 filters (ReLU) and a 2 × 2 `MaxPooling2D`. The code is the 7 × 7 × 8 array that comes out;
- `decoder`, the way back: a `Conv2D` of 8 filters (ReLU) and a 2 × 2 `UpSampling2D`, a `Conv2D` of 16 filters (ReLU) and a 2 × 2 `UpSampling2D`, then a last `Conv2D` with 1 filter and the sigmoid activation, which returns a 28 × 28 × 1 image;
- `autoencoder`, compiled with the Adam optimizer and the `"mse"` loss, like the dense denoisers, so that their errors compare.

Check the shapes with `autoencoder.summary(expand_nested=True)`, and compare its number of parameters with the dense denoisers'. Why not end the decoder with the `UpSampling2D`?

**Useful documentation**

- [`keras.layers.UpSampling2D`](https://keras.io/api/layers/reshaping_layers/up_sampling2d/)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `layers.Conv2D(16, 3, activation="relu", padding="same")`: `padding="same"` keeps the image's size; a small function `conv(filters, activation="relu")` that returns it saves repeating these arguments
- `layers.MaxPooling2D(2)` divides the height and the width by 2 (28 → 14 → 7), `layers.UpSampling2D(2)` multiplies them by 2 by repeating each pixel (7 → 14 → 28)
- the decoder takes `keras.Input(shape=(7, 7, 8))` inputs, the code's shape
- the last `Conv2D`: `conv(1, "sigmoid")`; an `UpSampling2D` at the end of the network would return blocks of 2 × 2 identical pixels, and an output without a sigmoid could leave [0, 1]
- `summary`: the encoder's output is `(None, 7, 7, 8)`, the decoder's `(None, 28, 28, 1)`; 3,217 parameters in all, against 9,420 + 10,192 = 19,612 for the dense denoiser with a code of 12 and 1,607,440 for part 3's: each 3 × 3 filter serves at every point of the image

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def conv(filters: int, activation: str = "relu") -> layers.Conv2D:
  return layers.Conv2D(filters, 3, activation=activation, padding="same")


encoder = keras.Sequential(
  [keras.Input(shape=(28, 28, 1)), conv(16), layers.MaxPooling2D(2), ..., ...],
  name="encoder",
)
decoder = keras.Sequential(
  [keras.Input(shape=(7, 7, 8)), conv(8), layers.UpSampling2D(2), ..., ..., ...],
  name="decoder",
)
autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(...)
autoencoder.summary(expand_nested=True)
```

#### Solution

In [ ]:
def conv(filters: int, activation: str = "relu") -> layers.Conv2D:
  return layers.Conv2D(filters, 3, activation=activation, padding="same")


encoder = keras.Sequential(
  [
    keras.Input(shape=(28, 28, 1)),
    conv(16),
    layers.MaxPooling2D(2),  # 14 × 14 × 16
    conv(8),
    layers.MaxPooling2D(2),  # 7 × 7 × 8, the code
  ],
  name="encoder",
)
decoder = keras.Sequential(
  [
    keras.Input(shape=(7, 7, 8)),
    conv(8),
    layers.UpSampling2D(2),  # 14 × 14 × 8
    conv(16),
    layers.UpSampling2D(2),  # 28 × 28 × 16
    # Smooths the UpSampling2D's blocks, output in ]0, 1[
    conv(1, "sigmoid"),
  ],
  name="decoder",
)
autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")  # like the dense denoisers
autoencoder.summary(expand_nested=True)

### Training

Only 5 epochs, against 20 and 10 for the dense denoisers: each one is much longer. The loss is the same, MSE: the errors compare.

In [ ]:
# 5 epochs: each one takes much longer than a dense denoiser's
autoencoder.fit(
  X_train_noisy_img,
  X_train_img,
  epochs=5,
  batch_size=256,
  validation_data=(X_test_noisy_img, X_test_img),
  verbose=2,
)
X_test_denoised = autoencoder.predict(X_test_noisy_img, verbose=0)
print(f"Error: {np.mean((X_test_denoised - X_test_img) ** 2):.4f}")
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

### On clean images

The denoiser has only seen noisy inputs. What does it return from a clean image?

In [ ]:
X_test_rebuilt = autoencoder.predict(X_test_img[:10], verbose=0)
show_images(X_test[:10], X_test_rebuilt, titles=titles)

On a clean image, the denoiser returns the garment's shape, but smoothed: it erases the fine details, like logos, checks or stripes, and forces the contrast. It learned to keep what looks like a garment and to erase the rest; to it, a fine detail looks like noise. Its error on the noisy images lies between those of the dense denoiser with a code of 12 and of the overcomplete denoiser, with 3,217 parameters instead of 19,612 and 1,607,440, and in 5 epochs: each 3 × 3 filter serves at every point of the image, whereas each weight of a dense layer only links one pixel to one unit.

## 5. A variational autoencoder

The previous notebook showed it: decoding a random code doesn't give a plausible image, as an autoencoder's codes take up only a small part of the latent space, with no known shape. A variational autoencoder (VAE) encodes each image into a distribution, $\mathcal{N}(\mu, \sigma^2)$, and a KL loss pushes these distributions towards $\mathcal{N}(0, I)$: one can then draw a code from $\mathcal{N}(0, I)$ and decode it.

What is specific to the VAE fits in one layer, `Sampling` (the "Reparameterization trick" and "Loss of a VAE" slides): write its `call` method, its `__init__` is given. It takes `[z_mean, z_log_var]`, $\mu$ and $\log \sigma^2$, two tensors of shape (batch size, `latent_dim`), and must:

- compute each example's KL, $-\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$, and add it to the model's loss with `self.add_loss`, averaged over the batch and multiplied by `self.beta`;
- draw $\varepsilon \sim \mathcal{N}(0, I)$, of `z_mean`'s shape, with the generator `self.seed_generator`, and return $z = \mu + \sigma \varepsilon$.

`beta` is 1 / 784 by default: the model's `"binary_crossentropy"` loss is a mean over the 784 pixels, whereas the KL is a sum over the code's dimensions. The cells after the solution build the VAE around `Sampling` and train it.

**Useful documentation**

- [`Layer.add_loss`](https://keras.io/api/layers/base_layer/#addloss-method)
- [`keras.random.normal`](https://keras.io/api/random/random_ops/#normal-function)
- [`keras.ops.sum`](https://keras.io/api/ops/numpy/#sum-function)
- [`keras.ops.mean`](https://keras.io/api/ops/numpy/#mean-function)
- [`keras.ops.square`](https://keras.io/api/ops/numpy/#square-function)
- [`keras.ops.exp`](https://keras.io/api/ops/numpy/#exp-function)

In [ ]:
class Sampling(layers.Layer):
  def __init__(self, beta: float = 1 / 784, **kwargs) -> None:
    super().__init__(**kwargs)
    self.beta = beta  # the KL's weight
    self.seed_generator = keras.random.SeedGenerator(0)

  def call(self, inputs):
    z_mean, z_log_var = inputs
    # Your code here
    ...

### Additional hints

The calls and parameters to use:

- `ops.exp(z_log_var)`: $\sigma^2$; `ops.exp(0.5 * z_log_var)`: $\sigma$; `ops.square(z_mean)`: $\mu^2$
- `ops.sum(..., axis=1)`: the sum over the code's dimensions, one KL per example; `ops.mean(kl)`: its mean over the batch
- `self.add_loss(self.beta * ops.mean(kl))`: Keras adds this term to the model's loss, at each training step
- `keras.random.normal(ops.shape(z_mean), seed=self.seed_generator)`: $\varepsilon$, a new draw at each call
- check: `s = Sampling(beta=1.0)`, then `s([ops.array([[1.0, 0.0]]), ops.zeros((1, 2))])`; `float(s.losses[0])` is then 0.5 (μ = (1, 0) and σ = 1: the KL is μ₁² / 2)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
class Sampling(layers.Layer):
  def __init__(self, beta: float = 1 / 784, **kwargs) -> None:
    super().__init__(**kwargs)
    self.beta = beta  # the KL's weight
    self.seed_generator = keras.random.SeedGenerator(0)

  def call(self, inputs):
    z_mean, z_log_var = inputs
    kl = -0.5 * ops.sum(..., axis=1)
    self.add_loss(...)
    eps = keras.random.normal(...)
    return ...
```

### Solution

In [ ]:
class Sampling(layers.Layer):
  def __init__(self, beta: float = 1 / 784, **kwargs) -> None:
    super().__init__(**kwargs)
    self.beta = beta  # the KL's weight
    self.seed_generator = keras.random.SeedGenerator(0)

  def call(self, inputs):
    z_mean, z_log_var = inputs
    kl = -0.5 * ops.sum(1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var), axis=1)
    self.add_loss(self.beta * ops.mean(kl))  # + the model's loss
    eps = keras.random.normal(ops.shape(z_mean), seed=self.seed_generator)
    return z_mean + ops.exp(0.5 * z_log_var) * eps  # z = μ + σ·ε

### The model

`build_vae(beta)` builds the VAE around `Sampling`, with a latent space of `latent_dim = 2` dimensions, so that it can be drawn:

- the encoder, with the functional API: an input of size `input_dim`, a `Dense` of 256 units (ReLU), then two `Dense` of `latent_dim` units without an activation, `z_mean` and `z_log_var`, and `z`, the output of `Sampling(beta)` on both; `encoder` is the `keras.Model` that returns `[z_mean, z_log_var, z]`;
- `decoder`, a `keras.Sequential`: a `Dense` of 256 units (ReLU), then the output of `input_dim` units with a sigmoid;
- `vae`, the `keras.Model` from the input to `decoder(z)`, compiled with Adam and the `"binary_crossentropy"` loss, to which `Sampling` adds the KL.

`vae.summary()` counts 404,244 parameters: `Sampling` has none.

In [ ]:
latent_dim = 2


def build_vae(beta: float = 1 / 784) -> tuple[keras.Model, keras.Model, keras.Model]:
  inputs = keras.Input(shape=(input_dim,))
  h = layers.Dense(256, activation="relu")(inputs)
  z_mean = layers.Dense(latent_dim)(h)
  z_log_var = layers.Dense(latent_dim)(h)  # log σ², any real number
  z = Sampling(beta)([z_mean, z_log_var])
  encoder = keras.Model(inputs, [z_mean, z_log_var, z], name="encoder")
  decoder = keras.Sequential(
    [
      keras.Input(shape=(latent_dim,)),
      layers.Dense(256, activation="relu"),
      layers.Dense(input_dim, activation="sigmoid"),
    ],
    name="decoder",
  )
  vae = keras.Model(inputs, decoder(z), name="vae")
  vae.compile(optimizer="adam", loss="binary_crossentropy")  # + KL
  return encoder, decoder, vae


encoder, decoder, vae = build_vae()
vae.summary()

### Training and latent space

The target is the clean image itself. Each point of the cloud is a test image's $\mu$, colored by its class.

In [ ]:
vae.fit(
  X_train,
  X_train,
  epochs=20,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=2,
)
test_means, _, _ = encoder.predict(X_test, verbose=0)
plt.figure(figsize=(7, 6))
for label, name in enumerate(class_names):
  points = test_means[y_test == label]
  plt.scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, label=name)
plt.legend(markerscale=5, fontsize=8)
plt.xlabel("$z_1$")
plt.ylabel("$z_2$")
plt.show()

Almost all the test images' μ stay between -3 and 3 on each axis, around the origin: the KL pulls each distribution towards $\mathcal{N}(0, I)$. Each class takes up its own region, without ever having seen a label. The shoes (sandals, sneakers, ankle boots) form a group apart, cut off from the clothes by an almost empty band; the trousers and the bags each have their own region. The tops (pullovers, coats, shirts) overlap: they look alike, and two dimensions aren't enough to separate them. Where each group lies changes from one training to the next: the latent space's axes have no fixed meaning.

## 6. Generating images

A VAE is a generative model: a code drawn from $\mathcal{N}(0, I)$, then decoded, must give a plausible image. The next cell draws 16 codes, `z_new`, and decodes them with `decoder`. Do the images look like clothes, unlike those decoded from noise in the previous notebook?

In [ ]:
z_new = keras.random.normal((16, latent_dim), seed=0)  # z ~ N(0, I)
generated = decoder.predict(z_new, verbose=0)
show_images(generated)

Most of them look like clothes: tops, trousers, shoes, blurry like every image of a two-dimensional VAE. A few faint ones mix two kinds of clothes: their code fell between two regions, where the decoder hesitates. In the previous notebook, a random code gave only blots: here, the KL put the training images' codes where $\mathcal{N}(0, I)$ draws its own, and the decoder knows how to decode them. A larger latent space would give sharper images, but one could no longer draw its map.

### The latent space's map

With 2 dimensions, one can decode a regular grid of codes, from -2 to 2 on each axis, where $\mathcal{N}(0, I)$ draws almost all its points: each cell shows what the decoder returns at that point. Compare with part 5's cloud.

In [ ]:
n = 12
grid = np.linspace(-2, 2, n)
# z_2 decreases from top to bottom, like the cloud's vertical axis
codes = np.array([[z1, z2] for z2 in grid[::-1] for z1 in grid])
images = decoder.predict(codes, verbose=0).reshape(n, n, 28, 28)
plt.figure(figsize=(8, 8))
plt.imshow(images.transpose(0, 2, 1, 3).reshape(n * 28, n * 28), cmap="gray_r")
plt.axis("off")
plt.show()

The map matches part 5's cloud: each kind of garment takes up the region of its μ, the shoes apart. From one cell to its neighbor, the image changes little: each point of the latent space decodes into an image, and close codes into close images. Between two regions, the images are faint and mixed.

## 7. The effect of β

The KL's weight `beta` sets a trade-off (the "Loss of a VAE" slide): without the KL, nothing brings the distributions close to $\mathcal{N}(0, I)$; the more the KL weighs, the closer the distributions get to $\mathcal{N}(0, I)$, and the less the code says about the image. Observe it:

- with `build_vae`, train two more VAEs, one with `beta=0`, the other with a β ten times larger than part 5's, `beta=10 / 784`, for 20 epochs each, with the same `fit` (and `verbose=0`);
- for each of the three β, with part 5's VAE, plot side by side the clouds of the test images' μ, colored by their class, and give the mean σ of their distributions and the mean squared reconstruction error of their decoded μ;
- show, for each β, the images its decoder returns from part 6's 16 codes `z_new`.

Look at the axes' scale. Where are the μ without the KL, and what is σ? Do the generated images still look like clothes? What is gained, and what is lost, with a β ten times larger?

**Useful documentation**

- [`matplotlib.pyplot.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `beta_encoder, beta_decoder, beta_vae = build_vae(0)`, then `beta_vae.fit(X_train, X_train, epochs=20, batch_size=256, verbose=0)`: a few minutes per VAE on a CPU
- a dictionary `models = {1 / 784: (encoder, decoder)}`, filled with the other two: `sorted(models.items())` goes through it by increasing β
- `_, axes = plt.subplots(1, 3, figsize=(18, 5))`: three plots side by side, where `ax.scatter` and `ax.set_title` replace `plt.scatter` and `plt.title`
- `beta_encoder(X_test)` calls the encoder directly and returns its three outputs, which `ops.convert_to_numpy` turns into NumPy arrays: the μ, the log σ² and the z; `np.exp(0.5 * log_vars).mean()`: the mean σ
- `beta_decoder(means)`: the decoded μ, to compare with `X_test`; `beta_decoder(z_new)`: the same 16 codes for each β

A skeleton to copy into the exercise cell, replacing each `...`:

```python
models = {1 / 784: (encoder, decoder)}
for beta in [0, 10 / 784]:
  beta_encoder, beta_decoder, beta_vae = build_vae(beta)
  beta_vae.fit(...)
  models[beta] = ...

_, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (beta, (beta_encoder, beta_decoder)) in zip(axes, sorted(models.items())):
  means, log_vars, _ = [ops.convert_to_numpy(t) for t in beta_encoder(X_test)]
  sigma = ...
  error = np.mean((ops.convert_to_numpy(beta_decoder(...)) - X_test) ** 2)
  for label, name in enumerate(class_names):
    points = ...
    ax.scatter(..., s=2, alpha=0.5, label=name)
  ax.set_title(f"β = {beta * 784:g} / 784: mean σ {sigma:.2f}, error {error:.4f}")
plt.show()

for beta, (_, beta_decoder) in sorted(models.items()):
  print(f"β = {beta * 784:g} / 784")
  show_images(...)
```

### Solution

In [ ]:
models = {1 / 784: (encoder, decoder)}  # part 5's VAE
for beta in [0, 10 / 784]:
  beta_encoder, beta_decoder, beta_vae = build_vae(beta)
  beta_vae.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=0,
  )
  models[beta] = beta_encoder, beta_decoder

_, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (beta, (beta_encoder, beta_decoder)) in zip(axes, sorted(models.items())):
  means, log_vars, _ = [ops.convert_to_numpy(t) for t in beta_encoder(X_test)]
  sigma = np.exp(0.5 * log_vars).mean()
  # The reconstruction error of the decoded μ
  error = np.mean((ops.convert_to_numpy(beta_decoder(means)) - X_test) ** 2)
  for label, name in enumerate(class_names):
    points = means[y_test == label]
    ax.scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, label=name)
  ax.set_title(f"β = {beta * 784:g} / 784: mean σ {sigma:.2f}, error {error:.4f}")
axes[2].legend(markerscale=5, fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

for beta, (_, beta_decoder) in sorted(models.items()):
  print(f"β = {beta * 784:g} / 784")
  show_images(ops.convert_to_numpy(beta_decoder(z_new)))

Without the KL (β = 0), the VAE falls back to an autoencoder: σ drops to 0, each image becomes a point again, and the μ spread far beyond [-3, 3], in groups separated by gaps. It rebuilds best, but no longer generates: the codes drawn from $\mathcal{N}(0, I)$ all fall in the small region around the origin, and the generated images all look alike, or like nothing. With a β ten times larger, the distributions widen (larger σ) and the μ huddle into a more regular cloud, where the tops overlap more: the reconstruction error goes up, the code says less about each image. The generated images stay plausible, smoother: with two dimensions, this β remains moderate; larger still, the classes would end up merging. Part 5's β is the loss of an ordinary VAE: the sum of the pixels' errors, plus the KL.